# Phase 3 — Data Cleaning & Transformation

This notebook begins Phase 3 with a pre-cleaning investigation of the raw hospital encounter data. No cleaning, transformation, or export is performed at this stage.

## 1. Imports and Raw Data Loading

In [2]:
import pandas as pd
from pathlib import Path

# Support running the notebook from either the project root or notebooks folder.
project_root = Path.cwd()
if not (project_root / "data" / "raw" / "diabetic_data.csv").exists():
    project_root = project_root.parent

raw_data_path = project_root / "data" / "raw" / "diabetic_data.csv"

# Load the raw file without changing its values. Literal question marks remain unchanged.
df_raw = pd.read_csv(raw_data_path)

print(f"Loaded raw data from: {raw_data_path}")
print(f"Dataset shape: {df_raw.shape}")

Loaded raw data from: /Users/josmithareddy/Documents/Data-Analyst-Projects/hospital-readmission-risk-analysis/data/raw/diabetic_data.csv
Dataset shape: (101766, 50)


## 2. Raw Data Integrity Check

These checks verify that this notebook starts from the same raw dataset examined during Phase 2.

In [3]:
total_rows, total_columns = df_raw.shape
unique_encounter_ids = df_raw["encounter_id"].nunique(dropna=False)
unique_patients = df_raw["patient_nbr"].nunique(dropna=False)
fully_duplicated_rows = df_raw.duplicated().sum()

print(f"Total rows: {total_rows:,}")
print(f"Total columns: {total_columns:,}")
print(f"Unique encounter IDs: {unique_encounter_ids:,}")
print(f"Unique patients: {unique_patients:,}")
print(f"Fully duplicated rows: {fully_duplicated_rows:,}")

Total rows: 101,766
Total columns: 50
Unique encounter IDs: 101,766
Unique patients: 71,518
Fully duplicated rows: 0


## 3. Missing Value Audit

The audit counts standard null/NaN values and literal `?` placeholders separately, then combines them for review. It does not replace either representation.

In [4]:
standard_null_counts = df_raw.isna().sum()
question_mark_counts = df_raw.eq("?").sum()
combined_missing_counts = standard_null_counts + question_mark_counts

missing_audit = pd.DataFrame({
    "column": df_raw.columns,
    "standard_null_count": standard_null_counts.values,
    "question_mark_count": question_mark_counts.values,
    "combined_missing_count": combined_missing_counts.values,
    "combined_missing_percentage": (
        combined_missing_counts.values / len(df_raw) * 100
    ),
})

# Keep only columns that contain at least one of the two missing-value forms.
missing_audit = (
    missing_audit.loc[missing_audit["combined_missing_count"] > 0]
    .sort_values("combined_missing_percentage", ascending=False)
    .reset_index(drop=True)
)

print("Missing-value audit:")
print(missing_audit.to_string(index=False, formatters={
    "combined_missing_percentage": lambda value: f"{value:.2f}%"
}))

Missing-value audit:
           column  standard_null_count  question_mark_count  combined_missing_count combined_missing_percentage
           weight                    0                98569                   98569                      96.86%
    max_glu_serum                96420                    0                   96420                      94.75%
        A1Cresult                84748                    0                   84748                      83.28%
medical_specialty                    0                49949                   49949                      49.08%
       payer_code                    0                40256                   40256                      39.56%
             race                    0                 2273                    2273                       2.23%
           diag_3                    0                 1423                    1423                       1.40%
           diag_2                    0                  358                     358

## 4. Investigate Question-Mark Column Discrepancy

Phase 2 contained a discrepancy: one detailed output appeared to show 6 columns containing `?`, while a later summary reported 7. The following fresh calculation reports the raw-data result directly without hiding or automatically reconciling that discrepancy.

In [5]:
question_mark_columns = question_mark_counts[question_mark_counts > 0].index.tolist()

question_mark_audit = pd.DataFrame({
    "column": question_mark_columns,
    "question_mark_count": question_mark_counts.loc[question_mark_columns].values,
    "question_mark_percentage": (
        question_mark_counts.loc[question_mark_columns].values / len(df_raw) * 100
    ),
}).sort_values("question_mark_percentage", ascending=False)

print(f"Columns containing at least one literal '?': {question_mark_columns}")
print(f"Total columns containing literal '?': {len(question_mark_columns)}")
print("Question-mark counts and percentages for every affected column:")
print(question_mark_audit.to_string(index=False, formatters={
    "question_mark_percentage": lambda value: f"{value:.2f}%"
}))

Columns containing at least one literal '?': ['race', 'weight', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3']
Total columns containing literal '?': 7
Question-mark counts and percentages for every affected column:
           column  question_mark_count question_mark_percentage
           weight                98569                   96.86%
medical_specialty                49949                   49.08%
       payer_code                40256                   39.56%
             race                 2273                    2.23%
           diag_3                 1423                    1.40%
           diag_2                  358                    0.35%
           diag_1                   21                    0.02%


## 5. High-Missingness Column Value Investigation

For this investigation only, temporary masks treat both standard null/NaN values and literal `?` placeholders as missing. The values in `df_raw` remain unchanged.

In [6]:
columns_to_investigate = [
    "weight",
    "medical_specialty",
    "payer_code",
    "race",
    "diag_2",
    "diag_3",
    "max_glu_serum",
    "A1Cresult",
]

for column in columns_to_investigate:
    series = df_raw[column]
    missing_mask = series.isna() | series.eq("?")
    observed_values = series.loc[~missing_mask]
    missing_percentage = missing_mask.mean() * 100
    unique_non_missing_values = observed_values.nunique()
    top_10_values = observed_values.value_counts().head(10)

    print(f"\nColumn: {column}")
    print(f"Missing percentage: {missing_percentage:.2f}%")
    print(f"Unique non-missing values: {unique_non_missing_values:,}")
    print("Top 10 observed non-missing values and counts:")
    print(top_10_values.to_string())


Column: weight
Missing percentage: 96.86%
Unique non-missing values: 9
Top 10 observed non-missing values and counts:
weight
[75-100)     1336
[50-75)       897
[100-125)     625
[125-150)     145
[25-50)        97
[0-25)         48
[150-175)      35
[175-200)      11
>200            3

Column: medical_specialty
Missing percentage: 49.08%
Unique non-missing values: 72
Top 10 observed non-missing values and counts:
medical_specialty
InternalMedicine              14635
Emergency/Trauma               7565
Family/GeneralPractice         7440
Cardiology                     5352
Surgery-General                3099
Nephrology                     1613
Orthopedics                    1400
Orthopedics-Reconstructive     1233
Radiologist                    1140
Pulmonology                     871

Column: payer_code
Missing percentage: 39.56%
Unique non-missing values: 17
Top 10 observed non-missing values and counts:
payer_code
MC    32439
HM     6274
SP     5007
BC     4655
MD     3532
CP     2

## 6. Cleaning Decision Table

This table documents the approved cleaning decisions. These decisions are recorded here only; no cleaning actions are executed in this section.

In [ ]:
# Document approved decisions only. This cell does not alter df_raw.
cleaning_decisions = pd.DataFrame([
    {
        "column": "gender",
        "issue": "Contains 3 'Unknown/Invalid' records",
        "missing_percentage": None,
        "proposed_action": "Remove the 3 records during cleaning",
        "reason": "The category is invalid and represents a negligible portion of the dataset.",
        "status": "Approved",
    },
    {
        "column": "weight",
        "issue": "96.86% missing",
        "missing_percentage": 96.86,
        "proposed_action": "Drop column",
        "reason": "Only approximately 3.14% of encounter records contain a recorded weight, making coverage too limited for the planned analysis.",
        "status": "Approved",
    },
    {
        "column": "medical_specialty",
        "issue": "49.08% missing represented by '?'",
        "missing_percentage": 49.08,
        "proposed_action": "Keep and convert missing placeholder to 'Unknown'",
        "reason": "Approximately half of the records contain specialty information and the variable may be useful for analyzing readmission patterns by provider specialty.",
        "status": "Approved",
    },
    {
        "column": "payer_code",
        "issue": "39.56% missing represented by '?'",
        "missing_percentage": 39.56,
        "proposed_action": "Drop column",
        "reason": "High missingness and limited relevance to the primary clinical readmission questions in this project.",
        "status": "Approved",
    },
    {
        "column": "race",
        "issue": "2.23% missing represented by '?'",
        "missing_percentage": 2.23,
        "proposed_action": "Keep and convert missing placeholder to 'Unknown'",
        "reason": "Most records contain race information and the field is useful for demographic analysis.",
        "status": "Approved",
    },
    {
        "column": "diag_1",
        "issue": "21 literal '?' values (0.02%)",
        "missing_percentage": 0.02,
        "proposed_action": "Replace '?' with standard missing value (NaN)",
        "reason": "Standardize the missing placeholder while keeping the remaining diagnosis codes unchanged at this stage.",
        "status": "Approved",
    },
    {
        "column": "diag_2",
        "issue": "358 literal '?' values (0.35%)",
        "missing_percentage": 0.35,
        "proposed_action": "Replace '?' with standard missing value (NaN)",
        "reason": "Standardize the missing placeholder while keeping the remaining diagnosis codes unchanged at this stage.",
        "status": "Approved",
    },
    {
        "column": "diag_3",
        "issue": "1,423 literal '?' values (1.40%)",
        "missing_percentage": 1.40,
        "proposed_action": "Replace '?' with standard missing value (NaN)",
        "reason": "Standardize the missing placeholder while keeping the remaining diagnosis codes unchanged at this stage.",
        "status": "Approved",
    },
    {
        "column": "admission_type_id",
        "issue": "No values outside the defined reference set",
        "missing_percentage": None,
        "proposed_action": "Retain all observed values as coded categories",
        "reason": "Treat as a coded categorical variable, not a continuous numeric variable.",
        "status": "Approved",
    },
    {
        "column": "discharge_disposition_id",
        "issue": "No values outside the defined reference set",
        "missing_percentage": None,
        "proposed_action": "Retain all observed values for now as coded categories",
        "reason": "Flag clinically or business-relevant disposition handling for later analysis; do not delete rare codes solely because of frequency.",
        "status": "Approved",
    },
    {
        "column": "admission_source_id",
        "issue": "No values outside the defined reference set",
        "missing_percentage": None,
        "proposed_action": "Retain all observed values as coded categories",
        "reason": "Treat as a coded categorical variable, not a continuous numeric variable.",
        "status": "Approved",
    },
    {
        "column": "insulin",
        "issue": "Observed categories: No, Steady, Down, and Up",
        "missing_percentage": None,
        "proposed_action": "Retain observed categories unchanged",
        "reason": "Retain No, Steady, Down, and Up in their original categorical form.",
        "status": "Approved",
    },
    {
        "column": "change",
        "issue": "Observed categories: No and Ch",
        "missing_percentage": None,
        "proposed_action": "Retain No and Ch unchanged",
        "reason": "Preserve the original categorical values.",
        "status": "Approved",
    },
    {
        "column": "diabetesMed",
        "issue": "Observed categories: Yes and No",
        "missing_percentage": None,
        "proposed_action": "Retain Yes and No unchanged",
        "reason": "Preserve the original categorical values.",
        "status": "Approved",
    },
    {
        "column": "readmitted",
        "issue": "Observed target categories: NO, >30, and <30",
        "missing_percentage": None,
        "proposed_action": "Retain original categories; do not convert to binary yet",
        "reason": "Binary 30-day readmission target creation will be handled in a later modeling or feature-engineering phase.",
        "status": "Approved",
    },
    {
        "column": "max_glu_serum",
        "issue": "94.75% standard missing values",
        "missing_percentage": 94.75,
        "proposed_action": "Keep as categorical and represent missing results as 'Not Recorded'",
        "reason": "Although sparsely populated, recorded glucose test results may contain useful information. Missing status must not be interpreted as a normal result, and observed laboratory categories will not be imputed.",
        "status": "Approved",
    },
    {
        "column": "A1Cresult",
        "issue": "83.28% standard missing values",
        "missing_percentage": 83.28,
        "proposed_action": "Keep as categorical and represent missing results as 'Not Recorded'",
        "reason": "A1C is relevant to diabetes-related analysis. Missing status must remain distinguishable from an actual normal test result, and observed laboratory categories will not be imputed.",
        "status": "Approved",
    },
])

print("Cleaning decision table:")
print(cleaning_decisions.to_string(index=False, formatters={
    "missing_percentage": lambda value: "N/A" if pd.isna(value) else f"{value:.2f}%"
}))

The categorical audit found no unexpected values outside the defined reference sets for `admission_type_id`, `discharge_disposition_id`, and `admission_source_id`. Rare coded categories are therefore not automatically considered data errors. Diagnosis `?` placeholders will be standardized to missing values during cleaning.

## 7. Categorical Data Audit

This section inspects important categorical variables before any cleaning or transformation is performed. All calculations read from `df_raw`; no values are replaced, removed, or changed.

### A. Inspect Core Categorical Variables

In [7]:
core_categorical_columns = ["gender", "age", "change", "diabetesMed", "readmitted"]

for column in core_categorical_columns:
    # Include every observed value, including standard missing values if present.
    counts = df_raw[column].value_counts(dropna=False)
    percentages = counts / len(df_raw) * 100
    value_summary = pd.DataFrame({
        "count": counts,
        "percentage_of_total": percentages,
    })

    print(f"\nColumn: {column}")
    print(f"Unique values: {df_raw[column].unique().tolist()}")
    print(value_summary.to_string(formatters={
        "percentage_of_total": lambda value: f"{value:.2f}%"
    }))


Column: gender
Unique values: ['Female', 'Male', 'Unknown/Invalid']
                 count percentage_of_total
gender                                    
Female           54708              53.76%
Male             47055              46.24%
Unknown/Invalid      3               0.00%

Column: age
Unique values: ['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)', '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']
          count percentage_of_total
age                                
[70-80)   26068              25.62%
[60-70)   22483              22.09%
[50-60)   17256              16.96%
[80-90)   17197              16.90%
[40-50)    9685               9.52%
[30-40)    3775               3.71%
[90-100)   2793               2.74%
[20-30)    1657               1.63%
[10-20)     691               0.68%
[0-10)      161               0.16%

Column: change
Unique values: ['No', 'Ch']
        count percentage_of_total
change                           
No      54755              53.80%

### B. Inspect Coded Hospital Variables

`admission_type_id`, `discharge_disposition_id`, and `admission_source_id` are categorical codes even though Pandas may store them as integers. This audit reports the observed codes without interpreting or mapping their numeric values.

In [8]:
coded_hospital_columns = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
]

for column in coded_hospital_columns:
    counts = df_raw[column].value_counts(dropna=False).sort_index()
    percentages = counts / len(df_raw) * 100
    code_summary = pd.DataFrame({
        "count": counts,
        "percentage_of_total": percentages,
    })

    print(f"\nColumn: {column}")
    print(f"Sorted unique values: {sorted(df_raw[column].dropna().unique().tolist())}")
    print(code_summary.to_string(formatters={
        "percentage_of_total": lambda value: f"{value:.2f}%"
    }))


Column: admission_type_id
Sorted unique values: [1, 2, 3, 4, 5, 6, 7, 8]
                   count percentage_of_total
admission_type_id                           
1                  53990              53.05%
2                  18480              18.16%
3                  18869              18.54%
4                     10               0.01%
5                   4785               4.70%
6                   5291               5.20%
7                     21               0.02%
8                    320               0.31%

Column: discharge_disposition_id
Sorted unique values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 22, 23, 24, 25, 27, 28]
                          count percentage_of_total
discharge_disposition_id                           
1                         60234              59.19%
2                          2128               2.09%
3                         13954              13.71%
4                           815               0.80%
5           

### C. Inspect Diagnosis Fields

Diagnosis codes are inspected in their raw form. No grouping, conversion, or other transformation is performed.

In [9]:
diagnosis_columns = ["diag_1", "diag_2", "diag_3"]

for column in diagnosis_columns:
    question_mark_count = df_raw[column].eq("?").sum()
    non_missing_mask = df_raw[column].notna() & df_raw[column].ne("?")
    top_20_codes = df_raw.loc[non_missing_mask, column].value_counts().head(20)

    print(f"\nColumn: {column}")
    print(f"Number of unique values: {df_raw[column].nunique(dropna=False):,}")
    print(f"Number of literal '?' values: {question_mark_count:,}")
    print("Top 20 most common non-missing codes:")
    print(top_20_codes.to_string())


Column: diag_1
Number of unique values: 717
Number of literal '?' values: 21
Top 20 most common non-missing codes:
diag_1
428      6862
414      6581
786      4016
410      3614
486      3508
427      2766
491      2275
715      2151
682      2042
434      2028
780      2019
996      1967
276      1889
38       1688
250.8    1680
599      1595
584      1520
V57      1207
250.6    1183
518      1115

Column: diag_2
Number of unique values: 749
Number of literal '?' values: 358
Top 20 most common non-missing codes:
diag_2
276       6752
428       6662
250       6071
427       5036
401       3736
496       3305
599       3288
403       2823
414       2650
411       2566
250.02    2074
707       1999
585       1871
584       1649
491       1545
250.01    1523
285       1520
780       1491
425       1434
682       1433

Column: diag_3
Number of unique values: 790
Number of literal '?' values: 1,423
Top 20 most common non-missing codes:
diag_3
250       11555
401        8289
276        5175

### D. Inspect Diabetes-Related Categorical Variables

The following value counts include standard missing values. Missing values are reported as observed and are not replaced.

In [10]:
diabetes_categorical_columns = [
    "insulin",
    "change",
    "diabetesMed",
    "max_glu_serum",
    "A1Cresult",
]

for column in diabetes_categorical_columns:
    print(f"\nColumn: {column}")
    print(df_raw[column].value_counts(dropna=False).to_string())


Column: insulin
insulin
No        47383
Steady    30849
Down      12218
Up        11316

Column: change
change
No    54755
Ch    47011

Column: diabetesMed
diabetesMed
Yes    78363
No     23403

Column: max_glu_serum
max_glu_serum
NaN     96420
Norm     2597
>200     1485
>300     1264

Column: A1Cresult
A1Cresult
NaN     84748
>8       8216
Norm     4990
>7       3812


### E. Audit Summary

The summary below flags observed representations for further review. A label is not classified as erroneous merely because it appears unusual. Unexpected-value checks are limited to the low-cardinality fields for which explicit reference sets are defined in the code.

In [11]:
# Combine the audited column lists while preserving their original order.
categorical_audit_columns = list(dict.fromkeys(
    core_categorical_columns
    + coded_hospital_columns
    + diagnosis_columns
    + diabetes_categorical_columns
))

columns_with_unknown_labels = [
    column for column in categorical_audit_columns
    if df_raw[column].isin(["Unknown", "Unknown/Invalid"]).any()
]
columns_with_question_marks = [
    column for column in categorical_audit_columns
    if df_raw[column].eq("?").any()
]
columns_with_standard_nan = [
    column for column in categorical_audit_columns
    if df_raw[column].isna().any()
]

# Reference sets support reporting only; they do not change or remove values.
expected_values = {
    "gender": {"Female", "Male", "Unknown/Invalid"},
    "age": {
        "[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)",
        "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)",
    },
    "change": {"Ch", "No"},
    "diabetesMed": {"Yes", "No"},
    "readmitted": {"NO", ">30", "<30"},
    "insulin": {"No", "Steady", "Up", "Down"},
    "max_glu_serum": {"Norm", ">200", ">300"},
    "A1Cresult": {"Norm", ">7", ">8"},
}

unexpected_values = {}
for column, expected in expected_values.items():
    observed = set(df_raw[column].dropna().unique())
    unexpected = sorted(observed - expected)
    if unexpected:
        unexpected_values[column] = unexpected

print("Categorical audit summary:")
print(f"Columns containing 'Unknown' or 'Unknown/Invalid': {columns_with_unknown_labels}")
print(f"Columns containing literal '?': {columns_with_question_marks}")
print(f"Columns containing standard NaN values: {columns_with_standard_nan}")
print(f"Values outside the defined reference sets (review only): {unexpected_values}")
print("Values reported for review are not automatically classified as erroneous.")

Categorical audit summary:
Columns containing 'Unknown' or 'Unknown/Invalid': ['gender']
Columns containing literal '?': ['diag_1', 'diag_2', 'diag_3']
Columns containing standard NaN values: ['max_glu_serum', 'A1Cresult']
Values outside the defined reference sets (review only): {}
Values reported for review are not automatically classified as erroneous.


## 7. Pre-Cleaning Investigation Summary

The raw dataset has not been modified. Missingness can be represented as either standard null/NaN values or literal `?` placeholders. High missingness alone is not sufficient reason to automatically drop a variable. Cleaning decisions will be made only after reviewing each field's analytical relevance and data quality.

The next step is to finalize the cleaning decision table.

# 8. Apply Approved Cleaning Decisions

Only the actions documented in the approved Cleaning Decision Table are applied below. The raw DataFrame remains unchanged, and diagnosis grouping, target creation, feature engineering, encoding, and scaling are deferred.

In [12]:
# Create a separate working copy so the raw DataFrame is never modified.
df_clean = df_raw.copy()

# Preserve a deep snapshot for a later raw-data integrity assertion.
df_raw_snapshot = df_raw.copy(deep=True)
original_row_count, original_column_count = df_raw.shape

print(f"Original row count: {original_row_count:,}")
print(f"Original column count: {original_column_count:,}")

Original row count: 101,766
Original column count: 50


In [13]:
# Record affected counts before applying any approved operation.
question_mark_columns_to_standardize = ["race", "diag_1", "diag_2", "diag_3"]
question_mark_counts_before = {
    column: int(df_clean[column].eq("?").sum())
    for column in question_mark_columns_to_standardize
}
invalid_gender_rows = int(df_clean["gender"].eq("Unknown/Invalid").sum())
medical_specialty_placeholders = int(df_clean["medical_specialty"].eq("?").sum())
max_glu_serum_missing = int(df_clean["max_glu_serum"].isna().sum())
a1cresult_missing = int(df_clean["A1Cresult"].isna().sum())

# Standardize approved literal question-mark placeholders to pandas missing values.
for column in question_mark_columns_to_standardize:
    df_clean[column] = df_clean[column].replace("?", pd.NA)

# Remove only the records with the approved invalid gender category.
df_clean = df_clean.loc[df_clean["gender"] != "Unknown/Invalid"].copy()

# Drop only the two columns approved for removal.
df_clean = df_clean.drop(columns=["weight", "payer_code"])

# Apply the approved categorical missing-value labels.
df_clean["medical_specialty"] = df_clean["medical_specialty"].replace("?", "Unknown")
df_clean["max_glu_serum"] = df_clean["max_glu_serum"].fillna("Not Recorded")
df_clean["A1Cresult"] = df_clean["A1Cresult"].fillna("Not Recorded")

print("Approved cleaning operations applied to df_clean only.")

Approved cleaning operations applied to df_clean only.


## 8.1 Cleaning Operation Log

The operation log records each approved action and the number of rows or values affected.

In [14]:
cleaning_operation_log = pd.DataFrame([
    {
        "operation": "Standardize '?' to pd.NA",
        "affected_column": column,
        "rows_or_values_affected": question_mark_counts_before[column],
        "reason": "Standardize the approved missing-value placeholder",
    }
    for column in question_mark_columns_to_standardize
] + [
    {
        "operation": "Remove rows",
        "affected_column": "gender",
        "rows_or_values_affected": invalid_gender_rows,
        "reason": "Remove the approved invalid 'Unknown/Invalid' category",
    },
    {
        "operation": "Drop column",
        "affected_column": "weight",
        "rows_or_values_affected": 1,
        "reason": "Coverage is too limited for the planned analysis",
    },
    {
        "operation": "Drop column",
        "affected_column": "payer_code",
        "rows_or_values_affected": 1,
        "reason": "High missingness and limited relevance to the primary questions",
    },
    {
        "operation": "Replace '?' with 'Unknown'",
        "affected_column": "medical_specialty",
        "rows_or_values_affected": medical_specialty_placeholders,
        "reason": "Retain specialty information with an explicit missing category",
    },
    {
        "operation": "Represent missing as 'Not Recorded'",
        "affected_column": "max_glu_serum",
        "rows_or_values_affected": max_glu_serum_missing,
        "reason": "Keep missing status distinct from observed laboratory results",
    },
    {
        "operation": "Represent missing as 'Not Recorded'",
        "affected_column": "A1Cresult",
        "rows_or_values_affected": a1cresult_missing,
        "reason": "Keep missing status distinct from observed laboratory results",
    },
])

print("Cleaning operation log:")
print(cleaning_operation_log.to_string(index=False))

Cleaning operation log:
                          operation   affected_column  rows_or_values_affected                                                          reason
           Standardize '?' to pd.NA              race                     2273              Standardize the approved missing-value placeholder
           Standardize '?' to pd.NA            diag_1                       21              Standardize the approved missing-value placeholder
           Standardize '?' to pd.NA            diag_2                      358              Standardize the approved missing-value placeholder
           Standardize '?' to pd.NA            diag_3                     1423              Standardize the approved missing-value placeholder
                        Remove rows            gender                        3          Remove the approved invalid 'Unknown/Invalid' category
                        Drop column            weight                        1                Coverage is too limited 

# 9. Post-Cleaning Validation

These checks compare the cleaned copy with the preserved raw DataFrame and verify the approved outcomes.

In [15]:
cleaned_row_count, cleaned_column_count = df_clean.shape
rows_removed = original_row_count - cleaned_row_count
columns_removed = original_column_count - cleaned_column_count
duplicate_encounter_id_count = int(df_clean["encounter_id"].duplicated().sum())
fully_duplicated_row_count = int(df_clean.duplicated().sum())

remaining_question_marks = df_clean.eq("?").sum()
remaining_question_marks = remaining_question_marks[remaining_question_marks > 0]
remaining_standard_missing = df_clean.isna().sum()
remaining_standard_missing = remaining_standard_missing[remaining_standard_missing > 0]

print(f"Original dataset shape: {df_raw.shape}")
print(f"Cleaned dataset shape: {df_clean.shape}")
print(f"Number of rows removed: {rows_removed:,}")
print(f"Number of columns removed: {columns_removed:,}")
print(f"Duplicate encounter_id count: {duplicate_encounter_id_count:,}")
print(f"Fully duplicated row count: {fully_duplicated_row_count:,}")
print("Remaining literal '?' counts by column:")
print(remaining_question_marks.to_string() if not remaining_question_marks.empty else "None")
print("Remaining standard missing values by column:")
print(remaining_standard_missing.to_string() if not remaining_standard_missing.empty else "None")
print(f"Unique gender values: {df_clean['gender'].unique().tolist()}")
print(f"Unique readmitted values: {df_clean['readmitted'].unique().tolist()}")
print(f"Unique encounter IDs: {df_clean['encounter_id'].nunique():,}")
print(f"Unique patients: {df_clean['patient_nbr'].nunique():,}")

# Explicit validation checks for the approved cleaning scope.
assert df_raw.shape == (101766, 50), "df_raw does not match the expected raw shape."
assert df_raw.equals(df_raw_snapshot), "df_raw was modified during cleaning."
assert not df_clean["gender"].eq("Unknown/Invalid").any(), "Invalid gender rows remain."
assert all(
    not df_clean[column].eq("?").any()
    for column in ["race", "diag_1", "diag_2", "diag_3"]
), "A literal '?' remains in an approved standardization column."
assert df_clean["encounter_id"].is_unique, "encounter_id is no longer unique."
assert set(df_clean["readmitted"].dropna().unique()) == {"NO", ">30", "<30"}, (
    "readmitted categories changed unexpectedly."
)

print("All post-cleaning validation assertions passed.")

Original dataset shape: (101766, 50)
Cleaned dataset shape: (101763, 48)
Number of rows removed: 3
Number of columns removed: 2
Duplicate encounter_id count: 0
Fully duplicated row count: 0
Remaining literal '?' counts by column:
None
Remaining standard missing values by column:
race      2271
diag_1      21
diag_2     358
diag_3    1423
Unique gender values: ['Female', 'Male']
Unique readmitted values: ['NO', '>30', '<30']
Unique encounter IDs: 101,763
Unique patients: 71,515
All post-cleaning validation assertions passed.


# 10. Save Cleaned Dataset

In [16]:
processed_directory = project_root / "data" / "processed"
processed_directory.mkdir(parents=True, exist_ok=True)

cleaned_output_path = processed_directory / "diabetic_data_cleaned.csv"
df_clean.to_csv(cleaned_output_path, index=False)

print(f"Cleaned dataset saved to: {cleaned_output_path.resolve()}")
print(f"Final row count: {len(df_clean):,}")
print(f"Final column count: {df_clean.shape[1]:,}")

Cleaned dataset saved to: /Users/josmithareddy/Documents/Data-Analyst-Projects/hospital-readmission-risk-analysis/data/processed/diabetic_data_cleaned.csv
Final row count: 101,763
Final column count: 48


# 11. Phase 3 Data Cleaning Summary

### Dataset Results

- **Starting dataset:** 101,766 rows, 50 columns, and 71,518 unique patients.
- **Final cleaned dataset:** 101,763 rows, 48 columns, 71,515 unique patients, and 101,763 unique encounter IDs.

### Cleaning Performed

- Removed 3 records where `gender` was `Unknown/Invalid`.
- Dropped `weight` because 96.86% of its values were missing.
- Dropped `payer_code` because 39.56% was missing and the field had limited relevance to the primary analysis.
- Standardized literal `?` placeholders to missing values in `race`, `diag_1`, `diag_2`, and `diag_3`.
- Represented missing `medical_specialty` placeholders as `Unknown`.
- Represented missing `max_glu_serum` and `A1Cresult` values as `Not Recorded`.

### Post-Cleaning Validation

- 0 duplicate encounter IDs and 0 fully duplicated rows remain.
- No literal `?` placeholders remain.
- `gender` contains only `Female` and `Male`.
- `readmitted` retains `NO`, `>30`, and `<30`.
- All validation assertions passed, and `df_raw` remained unchanged.

### Remaining Standard Missing Values

- `race`: 2,271
- `diag_1`: 21
- `diag_2`: 358
- `diag_3`: 1,423

These remaining race and diagnosis values are intentional standardized missing values and have not been imputed.

### Output and Deferred Work

The cleaned dataset was saved to `data/processed/diabetic_data_cleaned.csv`. Diagnosis grouping and feature engineering, binary 30-day readmission target creation, categorical encoding, scaling, and modeling were deliberately deferred to later phases.